In [1]:
# loading packages


import duckdb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import transformers
import accelerate
from transformers import BertForSequenceClassification
import time
import torch
import numpy as np
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification
import pickle
import os
import s3fs
from sklearn.metrics import classification_report
import tempfile


In [2]:
# load test data, nace, model and encoder

BUCKET = "thierry57"
MODEL_PATH_S3 = "model_nace"

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

# =========================
# load BERT model
# =========================

# load in temporary dir.

local_model_dir = tempfile.mkdtemp()

files_to_download = [
    "config.json",
    "model.safetensors",     
    "tokenizer.json",
    "tokenizer_config.json",
    "label_encoder.pkl",
    "test_df",
    "nace.pkl"
]

for file_name in files_to_download:
    s3_path = f"{BUCKET}/{MODEL_PATH_S3}/{file_name}"
    local_path = f"{local_model_dir}/{file_name}"

    with fs.open(s3_path, "rb") as s3_file:
        with open(local_path, "wb") as local_file:
            local_file.write(s3_file.read())


with open(f"{local_model_dir}/nace.pkl", "rb") as f:
    nace = pickle.load(f)

with open(f"{local_model_dir}/test_df", "rb") as f:
    test_df = pickle.load(f)    
# =========================
# load model + tokenizer
# =========================

model = BertForSequenceClassification.from_pretrained(local_model_dir)
model.eval()

tokenizer = BertTokenizer.from_pretrained(local_model_dir)

# =========================
# load label encoder
# =========================

with open(f"{local_model_dir}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# =========================
# Fonction tokenize
# =========================

def tokenize(batch):
    return tokenizer(
        batch['label'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [5]:
nace.head()

In [6]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [7]:

# conversion in datasets
test_dataset = Dataset.from_pandas(test_df[['label', 'target']])

test_dataset = test_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.rename_column("target", "labels")

test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


# temporary dataset reduced
#test_dataset = test_dataset.shuffle().select(range(4))


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    disable_tqdm=False,   # 🔥 important
    report_to="none"     
)
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

predictions on the test set

In [8]:
predictions = trainer.predict(test_dataset)

In [9]:
# accuracy on the test dataset

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))



The classification report shows that the imbalance between classes during training has a significant impact on classes for which there is little data. With more data on these classes, the results would probably have been better

In [10]:
# classification report
print(classification_report(y_true, y_pred))
y_pred_code = le.inverse_transform(y_pred)
y_true_code = le.inverse_transform(y_true)

In [11]:
# compare prediction and true class
df_results = test_df.copy().reset_index(drop=True)

df_results["true_code"] = y_true_code
df_results["pred_code"] = y_pred_code

I compare the predicted codes with the original codes and merge them with the NACE classification for greater clarity

In [12]:
nace = nace[['CODE', 'HEADING']]
df_results = df_results.merge(
    nace.rename(columns={"CODE": "true_code", "HEADING": "true_name"}),
    on="true_code",
    how="left"
)
df_results = df_results.merge(
    nace.rename(columns={"CODE": "pred_code",  "HEADING": "pred_name"}),
    on="pred_code",
    how="left"
)
df_results


I looked at the errors, and we can see that the predictions are often close 

In [13]:
#errors

df_errors = df_results[df_results["true_code"] != df_results["pred_code"]]
df_errors.head(20)

a simple function for making a prediction based on free-form text

In [15]:

def predict_label(text, model, tokenizer, le, nomenclature, device):
    # tokenisation
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # envoyer sur device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # prédiction
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_class_id = torch.argmax(logits, dim=1).cpu().numpy()[0]

    # décodage du label
    pred_code = le.inverse_transform([pred_class_id])[0]

    # récupérer le libellé associé
    pred_name = nace.loc[
        nace["CODE"] == pred_code, "HEADING"
    ].values[0]

    return {
        "text": text,
        "pred_class_id": pred_class_id,
        "pred_code": pred_code,
        "pred_name": pred_name
    }

    
# device (GPU si dispo)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [28]:
text = "someone who serves Big Macs at McDonald's "

result = predict_label(text, model, tokenizer, le, nace, device)

print("Text :", result["text"])
print("Classe prediction :", result["pred_class_id"])
print("Code prediction :", result["pred_code"])
print("name prediction :", result["pred_name"])

In [ ]:
df_errors.to_csv("./model_nace/df_errors.csv", index=False)